# Gold Layer - Olist E-commerce Dataset

**Objetivo:** Criação de tabelas analíticas agregadas e otimizadas para consumo de negócio seguindo a arquitetura Medalhão.

**Projetos e Entregas:**

### Projeto 1: Análise Comercial
* **Entrega 1:** `fat_vendas_comercial` - Métricas de vendas agregadas por período e categoria
* **Entrega 2:** Rankings de produtos (Top 5 mais vendidos e Top 5 menos vendidos)

### Projeto 2: Análise de Qualidade
* **Entrega 1:** `fat_avaliacoes_clientes` - Métricas de satisfação agregadas por categoria, vendedor e estado
* **Entrega 2:** Rankings de qualidade (produtos e vendedores melhor/pior avaliados)

**Características da Camada Gold:**
* Dados agregados e desnormalizados para performance
* Métricas de negócio pré-calculadas (KPIs)
* Otimizados para dashboards e relatórios executivos
* Redução de joins complexos para usuários finais
* Valores em múltiplas moedas (BRL e USD)

## 0. Configuração da Infraestrutura

Criação do schema Gold no Unity Catalog para armazenar tabelas analíticas agregadas.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS medalhao.gold;

## Projeto 1: Análise Comercial

### 1. Tabela Fato de Vendas Comerciais (fat_vendas_comercial)

Tabela analítica agregada por período (ano/mês) e categoria de produto com métricas de receita em BRL e USD.

In [0]:
from pyspark.sql import functions as F

# lendo as tabelas da camada silver
df_itens = spark.read.format("delta").table("medalhao.silver.fat_itens_pedidos")
df_pedidos = spark.read.format("delta").table("medalhao.silver.fat_pedido_total")
df_produtos = spark.read.format("delta").table("medalhao.silver.dim_produtos")
df_cotacao = spark.read.format("delta").table("medalhao.silver.dim_cotacao_dolar")

# cruzando os dados para ter a visão completa da venda no nível do item
df_base_vendas = (
    df_itens
    .join(df_pedidos, "id_pedido", "inner")
    .join(df_produtos, "id_produto", "left")
    .join(df_cotacao, F.to_date(df_pedidos.data_pedido) == df_cotacao.data_cotacao, "left")
)

# criando as agregações e métricas de negócio
df_fat_vendas_comercial = (
    df_base_vendas
    .withColumn("ano_venda", F.year("data_pedido"))
    .withColumn("mes_venda", F.month("data_pedido"))
    .groupBy("ano_venda", "mes_venda", "categoria_produto")
    .agg(
        F.countDistinct("id_pedido").alias("total_pedidos"),
        F.count("id_item").alias("qtd_itens_vendidos"),
        F.sum("preco_BRL").alias("receita_total_brl"),
        # calculando a receita em usd na hora da agregação
        F.sum(F.col("preco_BRL") / F.col("valor_cotacao")).alias("receita_total_usd") 
    )
    # calculando o ticket médio e arredondando as moedas
    .withColumn("ticket_medio_brl", F.round(F.col("receita_total_brl") / F.col("total_pedidos"), 2))
    .withColumn("receita_total_brl", F.round(F.col("receita_total_brl"), 2))
    .withColumn("receita_total_usd", F.round(F.col("receita_total_usd"), 2))
)

# salvando a tabela consolidada na camada gold 
(df_fat_vendas_comercial.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("medalhao.gold.fat_vendas_comercial"))

print("Tabela gold.fat_vendas_comercial criada!")
display(df_fat_vendas_comercial.limit(5))
print("-" * 50)

Tabela gold.fat_vendas_comercial criada!


ano_venda,mes_venda,categoria_produto,total_pedidos,qtd_itens_vendidos,receita_total_brl,receita_total_usd,ticket_medio_brl
2017,6,informatica_acessorios,219,261,37007.08,11250.77,168.98
2018,3,cama_mesa_banho,670,798,69256.39,21105.32,103.37
2017,3,cama_mesa_banho,249,289,25773.02,8239.87,103.51
2017,8,informatica_acessorios,297,350,35025.72,11118.94,117.93
2017,7,eletronicos,87,97,7346.84,2291.98,84.45


--------------------------------------------------


### 2. Rankings Comerciais - Top 5 Produtos

Análise de produtos mais e menos vendidos para identificação de oportunidades e produtos com baixa performance.

In [0]:
# agrupando no nível de produto para os rankings
df_ranking_produtos = (
    df_base_vendas
    .groupBy("nome_produto", "categoria_produto")
    .agg(F.count("id_item").alias("quantidade_vendida"))
    # filtrando nulos caso algum produto tenha vindo sem nome do dataset original
    .filter(F.col("nome_produto").isNotNull()) 
)

print("Top 5 Produtos MAIS Vendidos:")
df_top_5_mais = df_ranking_produtos.orderBy(F.col("quantidade_vendida").desc()).limit(5)
display(df_top_5_mais)

print("Top 5 Produtos MENOS Vendidos:")
df_top_5_menos = df_ranking_produtos.orderBy(F.col("quantidade_vendida").asc()).limit(5)
display(df_top_5_menos)

Top 5 Produtos MAIS Vendidos:


nome_produto,categoria_produto,quantidade_vendida
Secador de Cabelo,beleza_saude,1076
Toalha de Banho,cama_mesa_banho,919
Protetor Solar,beleza_saude,917
Travesseiro,cama_mesa_banho,839
Colar,relogios_presentes,804


Top 5 Produtos MENOS Vendidos:


nome_produto,categoria_produto,quantidade_vendida
Produto Genérico Preto,moveis_quarto,1
Item Básico Premium,fashion_calcados,1
Peça de Reposição Preto,telefonia_fixa,1
Item Básico Verde,industria_comercio_e_negocios,1
Peça de Reposição Plus,fashion_calcados,1


## Projeto 2: Análise de Qualidade

### 3. Tabela Fato de Avaliações de Clientes (fat_avaliacoes_clientes)

Tabela analítica agregada com métricas de satisfação por categoria, vendedor e estado.

In [0]:
from pyspark.sql import functions as F

# lendo as tabelas da silver
df_avaliacoes = spark.read.format("delta").table("medalhao.silver.fat_avaliacoes_pedidos")
df_itens = spark.read.format("delta").table("medalhao.silver.fat_itens_pedidos")
df_produtos = spark.read.format("delta").table("medalhao.silver.dim_produtos")
df_vendedores = spark.read.format("delta").table("medalhao.silver.dim_vendedores")

# cruzando os dados 
df_base_qualidade = (
    df_avaliacoes
    .join(df_itens, "id_pedido", "inner")
    .join(df_produtos, "id_produto", "left")
    .join(df_vendedores, "id_vendedor", "left")
)

# entrega 1: tabela principal (gold.fat_avaliacoes_clientes)
df_fat_avaliacoes_clientes = (
    df_base_qualidade
    .groupBy("categoria_produto", "nome_vendedor", "estado")
    .agg(
        F.count("id_avaliacao").alias("total_avaliacoes"),
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
        F.sum(F.when(F.col("nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
        F.sum(F.when(F.col("nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas")
    )
    # calculando o percentual com base nas agregações recém-criadas
    .withColumn("percentual_satisfacao", F.round((F.col("total_avaliacoes_positivas") / F.col("total_avaliacoes")) * 100, 2))
)

# salvando a tabela 
(df_fat_avaliacoes_clientes.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("medalhao.gold.fat_avaliacoes_clientes"))

print("Tabela gold.fat_avaliacoes_clientes criada!")
display(df_fat_avaliacoes_clientes.limit(5))
print("-" * 50)

Tabela gold.fat_avaliacoes_clientes criada!


categoria_produto,nome_vendedor,estado,total_avaliacoes,avaliacao_media,total_avaliacoes_positivas,total_avaliacoes_negativas,percentual_satisfacao
esporte_lazer,Emanuelly Mendonça,SP,410,4.13,324,56,79.02
ferramentas_jardim,Maya Cirino,SP,7,4.43,7,0,100.0
cool_stuff,Milena Moreira,RS,132,4.52,119,7,90.15
pet_shop,Bárbara Pacheco,RJ,98,4.2,78,18,79.59
brinquedos,Isabela Mendes,RJ,370,4.19,299,47,80.81


--------------------------------------------------


### 4. Rankings de Qualidade - Produtos e Vendedores

Identificação de produtos e vendedores melhor e pior avaliados para ações de reconhecimento e melhoria contínua.

In [0]:
# ranking de produtos
df_ranking_produtos_qualidade = (
    df_base_qualidade
    .filter(F.col("nome_produto").isNotNull())
    .groupBy("nome_produto", "categoria_produto")
    .agg(
        F.round(F.avg("nota_avaliacao"), 2).alias("nota_media"),
        F.count("id_avaliacao").alias("volume_avaliacoes")
    )
)

print("1. O Produto MAIS bem avaliado:")
# critério de desempate
display(df_ranking_produtos_qualidade.orderBy(F.col("nota_media").desc(), F.col("volume_avaliacoes").desc()).limit(1))

print("2. O Produto MENOS bem avaliado:")
display(df_ranking_produtos_qualidade.orderBy(F.col("nota_media").asc(), F.col("volume_avaliacoes").desc()).limit(1))

print("-" * 50)

# ranking de vendedores (agrupando por vendedor)
df_ranking_vendedores_qualidade = (
    df_base_qualidade
    .filter(F.col("nome_vendedor").isNotNull())
    .groupBy("nome_vendedor", "estado")
    .agg(
        F.round(F.avg("nota_avaliacao"), 2).alias("nota_media"),
        F.count("id_avaliacao").alias("volume_avaliacoes")
    )
)

print("3. O Vendedor MAIS bem avaliado:")
display(df_ranking_vendedores_qualidade.orderBy(F.col("nota_media").desc(), F.col("volume_avaliacoes").desc()).limit(1))

print("4. O Vendedor MENOS bem avaliado:")
display(df_ranking_vendedores_qualidade.orderBy(F.col("nota_media").asc(), F.col("volume_avaliacoes").desc()).limit(1))

1. O Produto MAIS bem avaliado:


nome_produto,categoria_produto,nota_media,volume_avaliacoes
Caneta Esferográfica Avançado,papelaria,5.0,18


2. O Produto MENOS bem avaliado:


nome_produto,categoria_produto,nota_media,volume_avaliacoes
Conjunto de Pincéis Ultra,artes,1.0,7


--------------------------------------------------
3. O Vendedor MAIS bem avaliado:


nome_vendedor,estado,nota_media,volume_avaliacoes
Luiz Otávio Abreu,PR,5.0,34


4. O Vendedor MENOS bem avaliado:


nome_vendedor,estado,nota_media,volume_avaliacoes
Sra. Fernanda Santos,SP,1.0,8
